<a href="https://colab.research.google.com/github/krasnovaov81-coder/netology/blob/main/itog_3_ipynb%22%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os

# Словарь, где ключи — пути к файлам, а значения — содержимое кода
project_files = {
    "config.py": '''# config.py

import os

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
DATA_DIR = os.path.join(BASE_DIR, "data")
REPORTS_DIR = os.path.join(BASE_DIR, "reports")
LOGS_DIR = os.path.join(BASE_DIR, "logs")
SRC_DIR = os.path.join(BASE_DIR, "src")

OUTPUT_REPORT_FILENAME = "summary_report.csv"
LOG_FILENAME = "errors.log"

STATUS_COLUMN = "status"
DELIVERED_STATUS_VALUE = "Delivered"
''',
    "src/analyzer.py": '''# src/analyzer.py

import pandas as pd
from config import (
    DATA_DIR,
    REPORTS_DIR,
    LOGS_DIR,
    OUTPUT_REPORT_FILENAME,
    LOG_FILENAME,
    STATUS_COLUMN,
    DELIVERED_STATUS_VALUE,
)
import os


class OrderAnalyzer:
    def __init__(self):
        self.results = []
        self.error_count = 0
        self.success_count = 0
        os.makedirs(REPORTS_DIR, exist_ok=True)
        os.makedirs(LOGS_DIR, exist_ok=True)
        self.log_path = os.path.join(LOGS_DIR, LOG_FILENAME)

    def _log_error(self, filename: str, error_msg: str):
        with open(self.log_path, "a", encoding="utf-8") as f:
            f.write(f"Файл: {filename} | Ошибка: {error_msg}\\n")

    def _process_file(self, filepath: str):
        filename = os.path.basename(filepath)
        try:
            df = pd.read_csv(filepath)
            if not {STATUS_COLUMN, "total_amount"}.issubset(df.columns):
                raise ValueError("Отсутствуют обязательные колонки")

            delivered_df = df[df[STATUS_COLUMN] == DELIVERED_STATUS_VALUE]
            if delivered_df.empty:
                return

            total_revenue = delivered_df["total_amount"].sum()
            average_check = delivered_df["total_amount"].mean()
            orders_count = len(delivered_df)

            self.results.append({
                "filename": filename,
                "total_revenue": round(total_revenue, 2),
                "average_check": round(average_check, 2),
                "orders_count": orders_count,
            })
            self.success_count += 1
        except Exception as e:
            self._log_error(filename, str(e))
            self.error_count += 1

    def process_all_files(self):
        if not os.path.exists(DATA_DIR):
            print(f"Папка '{DATA_DIR}' не найдена.")
            return
        for file in os.listdir(DATA_DIR):
            if file.lower().endswith(".csv"):
                full_path = os.path.join(DATA_DIR, file)
                self._process_file(full_path)

    def save_report(self):
        if not self.results:
            print("Нет данных для отчета.")
            return
        report_path = os.path.join(REPORTS_DIR, OUTPUT_REPORT_FILENAME)
        df_report = pd.DataFrame(self.results)
        df_report.to_csv(report_path, index=False)

    def print_summary(self):
        print(f"Обработано файлов: {self.success_count}")
        print(f"Файлов с ошибками: {self.error_count}")
''',
    "run.py": '''# run.py

from src.analyzer import OrderAnalyzer


def main():
    analyzer = OrderAnalyzer()
    analyzer.process_all_files()
    analyzer.save_report()
    analyzer.print_summary()


if __name__ == "__main__":
    main()
''',
    "requirements.txt": "pandas\n"
}

# Список папок для создания
folders_to_create = ["data", "reports", "logs", "src"]

print("Создаем структуру папок...")
for folder in folders_to_create:
    os.makedirs(folder, exist_ok=True)

print("Создаем и наполняем файлы кодом...")
for file_path, content in project_files.items():
    # Создаем вложенные папки, если они нужны (например, для src/)
    dir_name = os.path.dirname(file_path)
    if dir_name:
        os.makedirs(dir_name, exist_ok=True)

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(content)

print("Готово! Структура проекта создана.")
print("\nСодержимое корня:")
os.system("ls -l")
print("\nСодержимое папки src:")
os.system("ls -l src")

Создаем структуру папок...
Создаем и наполняем файлы кодом...
Готово! Структура проекта создана.

Содержимое корня:

Содержимое папки src:


0

In [2]:
!pip install pandas

In [3]:
!python run.py

Обработано файлов: 2
Файлов с ошибками: 1
